In [1]:
# Import FAISS vector store, Document class, and MultiQueryRetriever for multi-query semantic search

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.retrievers.multi_query import MultiQueryRetriever

In [2]:
# Add project root to sys.path and import custom functions to load Mistral LLM and Mini embeddings

import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
from utils.load_llms import load_mistral
from utils.load_embeddings import load_embedding_mini

In [3]:
# Create a list of Document objects covering health tips, energy, science, and technology topics
docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]


In [4]:
# Generate embeddings using custom Mini model and create a FAISS vector store from the documents

embedding_model = load_embedding_mini()  # Load embedding model
vectorstore = FAISS.from_documents(      # Create FAISS vector store for semantic search
    documents=docs,
    embedding=embedding_model
)

In [5]:
# Convert FAISS vector store into a similarity retriever to fetch top 5 similar documents

similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={'k': 5})

In [6]:
# Create a MultiQueryRetriever using Mistral LLM to expand queries and retrieve top 5 relevant documents

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k': 5}),
    llm=load_mistral()
)

In [7]:
# Run the query through both retrievers to get similarity-based and multi-query enhanced results

query = "How to improve energy levels and maintain balance?"

similarity_results = similarity_retriever.invoke(query)  # Retrieve top 5 similar documents
multiquery_results = multiquery_retriever.invoke(query)  # Retrieve documents using multi-query LLM expansion

In [8]:
# Print the content of documents retrieved by the similarity-based retriever

for i, doc in enumerate(similarity_results):
    print(f"\n--------- Result {i+1} ---------")
    print(doc.page_content)


--------- Result 1 ---------
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--------- Result 2 ---------
The solar energy system in modern homes helps balance electricity demand.

--------- Result 3 ---------
Consuming leafy greens and fruits helps detox the body and improve longevity.

--------- Result 4 ---------
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--------- Result 5 ---------
Photosynthesis enables plants to produce energy by converting sunlight.


In [9]:
# Print the content of documents retrieved by the multi-query retriever

for i, doc in enumerate(multiquery_results):
    print(f"\n--------- Result {i+1} ---------")
    print(doc.page_content)


--------- Result 1 ---------
The solar energy system in modern homes helps balance electricity demand.

--------- Result 2 ---------
Photosynthesis enables plants to produce energy by converting sunlight.

--------- Result 3 ---------
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--------- Result 4 ---------
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--------- Result 5 ---------
Regular walking boosts heart health and can reduce symptoms of depression.

--------- Result 6 ---------
Consuming leafy greens and fruits helps detox the body and improve longevity.

--------- Result 7 ---------
Deep sleep is crucial for cellular repair and emotional regulation.
